# Pytorch batched training


Explore some more features on the (very small) Diabetes dataset: Mainly batched training, 

In [1]:
# complete imports if needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start(

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

False


AssertionError: Torch not compiled with CUDA enabled

In [3]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

(442, 1)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Use the dataloader to split the dataset into smaller batches (have a look how different batch sizes compare). Hint: Don't go for too small batch sizes, it will massively slow down the training!

In [5]:
from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            dtype=torch.float32 # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            dtype=torch.float32 # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)

In [ ]:
train_dataset = MyDataFromDF(X_train, y_train)
test_dataset  = MyDataFromDF(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

Define the NN according to your first attempts.

In [7]:
class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, drop_out):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.dropout1 = nn.Dropout(p=drop_out) # only parts of the neurons are used.

        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.dropout2 = nn.Dropout(p=drop_out)

        self.fc3 = nn.Linear(hidden_size, int(hidden_size/2))  # one hidden layer, try to add another one
        self.dropout3 = nn.Dropout(p=drop_out)

        self.fc4 = nn.Linear(int(hidden_size/2), output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = self.dropout1(x)

        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        x = self.dropout2(x)

        x = torch.relu(self.fc3(x))  # ReLU activation function for hidden layer
        x = self.dropout3(x)
        
        x = self.fc4(x)   # no activation function for the output layer (because it is a regresion model!)
        return x

In [8]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 100
output_size = 1
drop_out=0.2 # specifies how much of the NN remains randomly inactive

learning_rate = 0.005

# Number of training iterations
num_epochs = 300

In [ ]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size, drop_out) # .to(device) # move model to GPU if available

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

Train your model. Note how the defined batches need some adjustments in the code (evaluating loss...)

In [12]:
# batched training
for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    total_samples = 0

    for data, targets in train_loader:
        data = data # .to(device)
        targets = targets # .to(device)

        outputs = model(data.to(torch.float32))

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [10/300], Loss: 4020.2023
Epoch [20/300], Loss: 4135.1557
Epoch [30/300], Loss: 3227.0574
Epoch [40/300], Loss: 3475.4416
Epoch [50/300], Loss: 3343.7125
Epoch [60/300], Loss: 3927.7858
Epoch [70/300], Loss: 3632.8439
Epoch [80/300], Loss: 3275.6526
Epoch [90/300], Loss: 3181.6551
Epoch [100/300], Loss: 3082.3957
Epoch [110/300], Loss: 3565.7056
Epoch [120/300], Loss: 3367.5926
Epoch [130/300], Loss: 3748.0096
Epoch [140/300], Loss: 3322.3301
Epoch [150/300], Loss: 3509.0513
Epoch [160/300], Loss: 3301.2397
Epoch [170/300], Loss: 3286.5495
Epoch [180/300], Loss: 3361.7867
Epoch [190/300], Loss: 3101.5652
Epoch [200/300], Loss: 3264.0119
Epoch [210/300], Loss: 3410.1190
Epoch [220/300], Loss: 3440.2228
Epoch [230/300], Loss: 3249.1493
Epoch [240/300], Loss: 3610.4302
Epoch [250/300], Loss: 3355.5615
Epoch [260/300], Loss: 3184.6503
Epoch [270/300], Loss: 3269.1067
Epoch [280/300], Loss: 3363.9920
Epoch [290/300], Loss: 3353.7508
Epoch [300/300], Loss: 3204.5777


Evaluate the model using the test set and see if the NN shows over or underfitting. Adjust the model accordingly or train again.

In [14]:
# Evaluate the model
model.eval()  # set model to eval mode (e.g. no dropout)

all_preds = []
all_targets = []

total_samples = 0
total_loss = 0

criterion = nn.MSELoss()

with torch.no_grad():
    for data, targets in test_loader:
        data = data # .to(device)
        targets = targets # .to(device)

        outputs = model(data.to(torch.float32))
        loss = criterion(outputs, targets)

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size # multiply by batch size to 
        total_samples += targets.size(0)
        
        # Predictions      
        probs = torch.sigmoid(outputs)     # convert for metrics
        preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().tolist()) # cpu() moves the tensor to cpu (if on gpu), tolist converts it to python list
        all_targets.extend(targets.cpu().tolist())

avg_loss = total_loss / total_samples

print(f"Average test Loss: {avg_loss:.4f}")


Average test Loss: 2580.8719


MD: already better resutls with a very simple model than without splitting, and still a little underfitting with the batch sizing.